# 第8章 文埋め込み

## 8.3 文埋め込みモデルの実装

### 8.3.2 教師ありのSimCSEの実装

SimCSE (Simple Contrastive learning of Sentence Embedding)

Contrastive learning: 対照学習

#### 準備

In [ ]:
# 必要なパッケージをインストールする
!pip install 'datasets<4.0.0' scipy 'transformers[ja,torch]<4.41.0'

In [ ]:
!pip install "peft<0.12"

In [ ]:
from transformers.trainer_utils import set_seed

set_seed(42)

#### データセットの読み込みと前処理

In [ ]:
from datasets import load_dataset

# Hugging Face Hubのllm-book/jsnliのリポジトリから
# JSNLIの訓練セットを読み込む
jsnli_dataset = load_dataset("llm-book/jsnli", split="train")

In [ ]:
# 訓練セットの形式と事例数を確認する
print(jsnli_dataset)

In [ ]:
from pprint import pprint

# JSNLIの訓練セットの内容を確認する
pprint(jsnli_dataset[0])
pprint(jsnli_dataset[1])

In [ ]:
import csv
import random
from typing import Iterator

# JSNLIの訓練セットから、前提文とラベルごとに仮説分をまとめたdictを作成する
premise2hypotheses = {}

premises = jsnli_dataset["premise"]  # 前提文
hypotheses = jsnli_dataset["hypothesis"]  # 仮説文
labels = jsnli_dataset["label"]  # ラベル

for premise, hypothesis, label in zip(premises, hypotheses, labels):
    if premise not in premise2hypotheses:
        premise2hypotheses[premise] = {
            "entailment": [],
            "neutral": [],
            "contradiction": [],
        }

    premise2hypotheses[premise][label].append(hypothesis)

In [ ]:

from datasets import Dataset

def generate_sup_train_example() -> Iterator[dict[str, str]]:
    """教師ありSimCSEの訓練セットの事例を生成"""
    # JSNLIのデータから（前提文、「含意」ラベルの仮説文、「矛盾」ラベルの仮説文）の3つ組を生成する
    for premise, hypotheses in premise2hypotheses.items():
        # 「矛盾」ラベルの仮説文が一つもない場合はスキップする
        if len(hypotheses["contradiction"]) == 0:
            continue

        # 「含意」ラベルの仮説文一つにつき、「矛盾」ラベルの仮説文一つをランダムに関連付ける
        for entailment_hypothesis in hypotheses["entailment"]:
            contradiction_hypothsis = random.choice(
                hypotheses["contradiction"]
            )
            # （前提文、「含意」ラベルの仮説文、「矛盾」ラベルの仮説文）の3つ組をdictとして生成
            yield {
                "premise": premise,
                "entailment_hypothesis": entailment_hypothesis,
                "contradiction_hypothesis": contradiction_hypothsis,
            }

# 定義したジェネレータ関数を用いて、教師ありSimCSEの訓練セットを構築する
sup_train_dataset = Dataset.from_generator(generate_sup_train_example)

In [ ]:
# 訓練セットの形式と事例数を確認する
print(sup_train_dataset)

In [ ]:
# 訓練セットの内容を確認する
pprint(sup_train_dataset[0])
pprint(sup_train_dataset[1])

#### トークナイザの準備

In [ ]:
from transformers import AutoTokenizer

# Hugging Face Hubにおけるモデル名を指定する
base_model_name = "cl-tohoku/bert-base-japanese-v3"
# モデル名からトークナイザを初期化する
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

#### collate関数の準備

In [ ]:
def sup_train_collate_fn():
    pass

In [ ]:
import torch
from torch import Tensor
from transformers import BatchEncoding

def eval_collate_fn(
    examples: list[dict],
) -> dict[str, BatchEncoding | Tensor]:
    """SimCSEの検証・テストセットのミニバッチを作成"""
    # ミニバッチの文ペアに含まれる文（文1と文2）のそれぞれに
    # トークナイザを適用する
    tokenized_texts_1 = tokenizer(
        [example["sentence1"] for example in examples],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    tokenized_texts_2 = tokenizer(
        [example["sentence2"] for example in examples],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )

    # 文1と文2の類似度行列における正例ペアの位置を示すTensorを作成する
    # 行列のi行目の事例（文1）に対して
    # i列目の事例（文2）との組が正例ペアとなる
    labels = torch.arange(len(examples))

    # データセットに付与された類似度スコアのTensorを作成する
    label_scores = torch.tensor(
        [example["label"] for example in examples]
    )

    return {
        "tokenized_texts_1": tokenized_texts_1,
        "tokenized_texts_2": tokenized_texts_2,
        "labels": labels,
        "label_scores": label_scores,
    }

#### モデルの準備

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers.utils import ModelOutput

class SimCSEModel(nn.Module):
    """SimCSEのモデル"""

    def __init__(
        self,
        base_model_name: str,
        mlp_only_train: bool = False,  # MLP: 多層パーセプトロン
        temperture: float = 0.5
    ):
        """モデルの初期化"""
        super().__init__()

        # モデル名からエンコーダを初期化する
        self.encoder = AutoModel.from_pretrained(base_model_name)
        # パラメータをメモリ上に隣接した形で配置
        # これを実行しない場合、モデルの保存でエラーになることがある
        for param in self.encoder.parameters():
            param.data = param.data.contiguous()

        # MLPの次元数
        self.hidden_size = self.encoder.config.hidden_size
        # MLP層の線形層
        self.dense = nn.Linear(self.hidden_size, self.hidden_size)
        # MLP層の活性化関数
        self.activation = nn.Tanh()

        # MLP層による変換を訓練時にのみ適用するよう設定するフラグ
        self.mlp_only_train = mlp_only_train
        # 交差エントロピー損失の計算時に使用する温度
        self.temperture = temperture

    def encode_texts(self, tokenized_texts: BatchEncoding) -> Tensor:
        """エンコーダを用いて文をベクトルに変換"""
        # トークナイズされた文をエンコーダに入力する
        encoded_texts = self.encoder(**tokenized_texts)
        # モデルの最終層の出力(last_hidden_state)の
        # [CLS]トークン(0番目の位置のトークン)のベクトルを取り出す
        encoded_texts = encoded_texts.last_hidden_state[:,0]

        # self.mlp_only_trainのフラグがTrueに設定されていて、
        # かつ訓練時でない場合、MLP層の変換を適用せずにベクトルを返す
        if self.mlp_only_train and not self.training:
            return encoded_texts

        # MLP層におけるベクトルの変換を行う
        encoded_texts = self.dense(encoded_texts)
        encoded_texts = self.activation(encoded_texts)

        return encoded_texts

    def forward(
        self,
        tokenized_texts_1: BatchEncoding,
        tokenized_texts_2: BatchEncoding,
        labels: Tensor,
        label_scores: Tensor | None = None
    ) -> ModelOutput:
        """モデルの前向き計算を定義"""
        # 文ペアをベクトルに変換する
        encoded_texts_1 = self.encode_texts(tokenized_texts_1)
        encoded_texts_2 = self.encode_texts(tokenized_texts_2)

        # 文ペアの類似度行列を作成する
        sim_matrix = F.cosine_similarity(
            encoded_texts_1.unsqueeze(1),
            encoded_texts_2.unsqueeze(0),
            dim=2
        )

        # 交差エントロピー損失を求める
        loss = F.cross_entropy(sim_matrix / self.temperture, labels)

        # 性能評価に使用するため、正例ペアに対するスコアを類似度行列から取り出す
        positive_mask = F.one_hot(labels, sim_matrix.size(1)).bool()
        positive_scores = torch.masked_select(
            sim_matrix, positive_mask
        )

        return ModelOutput(loss=loss, scores=positive_scores)

In [ ]:
# 教師ありSimCSEのモデルを初期化する
sup_model = SimCSEModel(base_model_name, mlp_only_train=False)

#### 検証セット、テストセットの準備

In [ ]:
# Hugging Face Hubのllm-book/JGLUEのリポジトリから
# JSTSデータセットの訓練セットと検証セットを読み込み、
# それぞれをSimCSEの検証セットとテストセットとして使用する
# JSTSはテストセットが公開されていないため、
# 検証セット -> "train", テストセット -> "validation"で対応
valid_dataset = load_dataset(
    "llm-book/JGLUE", name="JSTS", split="train"
)
test_dataset = load_dataset(
    "llm-book/JGLUE", name="JSTS", split="validation"
)

#### `Trainer`の準備

In [ ]:
from scipy.stats import spearmanr
from transformers import EvalPrediction

def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    """モデルが予測したスコアと評価用データのスコアのスピアマンの順位相関係数を計算"""
    scores = p.predictions
    labels, label_scores = p.label_ids

    spearman = spearmanr(scores, label_scores).statistic

    return {"spearman": spearman}

In [ ]:
from transformers import TrainingArguments

# 教師なしSimCSEの訓練のハイパーパラメータを設定する
sup_training_args = TrainingArguments(
    output_dir="outputs_sup_simcse",  # 結果の保存先フォルダ
    per_device_train_batch_size=128,     # 訓練時のバッチサイズ
    per_device_eval_batch_size=128,      # 評価時のバッチサイズ
    learning_rate=5e-5,                 # 学習率
    num_train_epochs=3,                 # 訓練エポック数
    evaluation_strategy="steps",        # 検証セットによる評価のタイミング
    eval_steps=250,                     # 検証セットによる評価を行う訓練ステップ数の間隔
    logging_steps=250,                  # ロギングを行う訓練ステップ数の間隔
    save_steps=250,                     # チェックポイントを保存する訓練ステップ数の間隔
    save_total_limit=1,                 # 保存するチェックポイントの最大数
    fp16=True,                          # 自動混合精度演算の有効化
    load_best_model_at_end=True,        # 最良のモデルを訓練終了後に読み込むか
    metric_for_best_model="spearman",   # 最良のモデルを決定する評価指標
    remove_unused_columns=False,        # データセットの不要フィールドを削除するか
)

In [ ]:
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import Trainer

class SimCSETrainer(Trainer):
    """SimCSEの訓練に使用するTrainer"""

    def get_eval_dataloader(
        self, eval_dataset: Dataset | None = None
    ):
        """
        検証・テストセットのDataLoaderでeval_collate_fnを使うように
        Trainerのget_eval_dataloaderをオーバーライド
        """
        if eval_dataset is None:
            eval_dataset = self.eval_dataset

        return DataLoader(
            eval_dataset,
            batch_size=64,
            collate_fn=eval_collate_fn,
            pin_memory=True,
        )

# 教師なしSimCSEのTrainerを初期化する
sup_trainer = SimCSETrainer(
    model=sup_model,
    args=sup_training_args,
    data_collator=sup_train_collate_fn,
    train_dataset=sup_train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
)

#### 訓練の実行

In [ ]:
# 教師なしSimCSEの訓練を行う
sup_trainer.train()

#### 性能評価

In [ ]:
# 検証セットで教師なしSimCSEのモデルの評価を行う
sup_trainer.evaluate(valid_dataset)

In [ ]:
# テストセットで教師なしSimCSEのモデルの評価をおこなう
sup_trainer.evaluate(test_dataset)

#### トークナイザとモデルの保存

#### Google Driveへの保存

In [ ]:
from google.colab import drive

drive.mount("drive")